# Poverty Map (Google Earth Engine)

In [ ]:
### The VIIRS DNB dataset has 15 arcsec (about 463m at the equator) resolution nighttime luminosity. 
### We compute six aggregate summary statistics6 of the night light intensities within 1 square mile (1.6x1.6km2), 5x5km2, and 10x10km2 areas around each populated place.
### --> Max/mean/median luminosity, ratio of zero luminosity, average of upper/lower third luminosity.

## Setup

### Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
!ln -s /content/drive/MyDrive/Colab\ Notebooks/SES-Inference /mydrive

Mounted at /content/drive


### Google Earth Engine

In [ ]:
import ee
ee.Authenticate()
ee.Initialize()

### Dependencies

In [ ]:
!pip install geopandas

In [6]:
### System
import sys
import itertools
from tqdm import tqdm

### Local
sys.path.append('/mydrive/libs/')
%set_env PYTHONPATH=/env/python:/mydrive/libs/

%load_ext autoreload
%autoreload 2

from maps import geo
from utils import ios
from maps.viirs import VIIRS

env: PYTHONPATH=/env/python:/mydrive/libs/
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Survey data (clusters)

In [7]:
### by DHS cluster
fn = '/mydrive/data/Uganda/results/DHS2016_MIS2018_iwi_cluster.csv'
df_dhs = ios.load_csv(fn, index_col=0)
df_dhs.head()

,DHSCC,DHSYEAR,DHSCLUST,URBAN_RURA,LATNUM,LONGNUM,SOURCE,ALT_GPS,ALT_DEM,DATUM,mean_iwi,SES
0,UG,2016,1,1,0.320188,32.568206,GPS,9999.0,1197.0,WGS84,43.300000,rich
1,UG,2016,2,1,0.340653,32.593627,GPS,9999.0,1179.0,WGS84,24.967857,rich
2,UG,2016,3,1,0.313103,32.566556,GPS,9999.0,1189.0,WGS84,34.732000,rich
3,UG,2016,4,1,0.353368,32.558144,GPS,9999.0,1181.0,WGS84,36.060714,rich
4,UG,2016,5,1,0.367388,32.594357,GPS,9999.0,1226.0,WGS84,44.857692,rich


## Nightlight luminosity 

### Examples

In [8]:
year = 2016
rad_gte_thres = 1.0

points = []
points.append([32.02786014765624,0.4653902099549784]) # Kampala (dark)
points.append([32.15326611328126,0.37593062258811094]) # Kampala (grey)
points.append([32.046836059570325,0.40373912169855747]) # Kampala (white)
points.append([8.66414963290783,50.118508372433]) # Frankfurt (white)
points.append([-73.97282574075544,40.779868876549926]) # Manhattan (white)

nl = VIIRS(source="viirs")
nl.filter(select='mean', date_start='{}-01-01'.format(year), date_end='{}-01-01'.format(year+1))
df = nl.fast_lights_at_points(points=points, buffer_meters=geo.KM_TO_M, rad_gte_thres=rad_gte_thres)
df.head(10)

,NTLL_1.0km_min,NTLL_1.0km_max,NTLL_1.0km_mean,NTLL_1.0km_median,NTLL_1.0km_l3_mean,NTLL_1.0km_u3_mean,NTLL_1.0km_rad_gte_1.0_area,NTLL_1.0km_rad_gte_1.0_pixels,NTLL_1.0km_rad_gte_1.0_sum_rad
0,0.014531,0.691798,0.124909,0.060224,0.042721,0.267325,0.0,0.0,0.0
1,0.042873,0.490402,0.160235,0.143833,0.091060,0.242921,0.0,0.0,0.0
2,1.471329,21.682526,9.868459,9.745038,3.560544,17.396157,1.0,1.0,1.0
3,19.161980,177.545059,62.496984,40.741665,27.166705,111.527874,1.0,1.0,1.0
4,30.965986,111.024269,46.646710,46.806850,33.904157,60.700114,1.0,1.0,1.0


### All-in-one (fast)

In [9]:
columns = ['DHSYEAR','LONGNUM','LATNUM']
df_dhs = df_dhs[columns].copy()
years = df_dhs.DHSYEAR.unique()
meters = geo.METERS
total = len(years)*len(meters)
rad_gte_thres = 10.0
newyear = -1

for year, meters in tqdm(itertools.product(years,meters), total=total):

  # constructor per year
  if newyear != year:
    nl = VIIRS(source="viirs")
    nl.filter(select='mean', date_start='{}-01-01'.format(year), date_end='{}-01-01'.format(year+1))
    newyear = year
    
  # points
  index = df_dhs.query("DHSYEAR==@year").index.values
  points = df_dhs.loc[index,['LONGNUM','LATNUM']].values
  df = nl.fast_lights_at_points(points=points, buffer_meters=meters, rad_gte_thres=rad_gte_thres)
  df.set_index(index, inplace=True)
  df_dhs.loc[index,df.columns.values] = df

100%|██████████| 8/8 [06:53<00:00, 51.69s/it]


In [ ]:
df_dhs = df_dhs[columns].copy()
df_dhs.drop(columns=columns, inplace=True)
df_dhs.sample(10)

,DHSCC,DHSYEAR,DHSCLUST,URBAN_RURA,LATNUM,LONGNUM,SOURCE,ALT_GPS,ALT_DEM,DATUM,mean_iwi,SES,NTLL_1.6km_min,NTLL_1.6km_max,NTLL_1.6km_mean,NTLL_1.6km_median,NTLL_1.6km_l3_mean,NTLL_1.6km_u3_mean,NTLL_1.6km_rad_gte_10.0_area,NTLL_1.6km_rad_gte_10.0_pixels,NTLL_1.6km_rad_gte_10.0_sum_rad,NTLL_2.0km_min,NTLL_2.0km_max,NTLL_2.0km_mean,NTLL_2.0km_median,NTLL_2.0km_l3_mean,NTLL_2.0km_u3_mean,NTLL_2.0km_rad_gte_10.0_area,NTLL_2.0km_rad_gte_10.0_pixels,NTLL_2.0km_rad_gte_10.0_sum_rad,NTLL_5.0km_min,NTLL_5.0km_max,NTLL_5.0km_mean,NTLL_5.0km_median,NTLL_5.0km_l3_mean,NTLL_5.0km_u3_mean,NTLL_5.0km_rad_gte_10.0_area,NTLL_5.0km_rad_gte_10.0_pixels,NTLL_5.0km_rad_gte_10.0_sum_rad,NTLL_10.0km_min,NTLL_10.0km_max,NTLL_10.0km_mean,NTLL_10.0km_median,NTLL_10.0km_l3_mean,NTLL_10.0km_u3_mean,NTLL_10.0km_rad_gte_10.0_area,NTLL_10.0km_rad_gte_10.0_pixels,NTLL_10.0km_rad_gte_10.0_sum_rad
526,UG,2016,528,2,0.658938,30.176259,GPS,9999.0,1650.0,WGS84,17.643333,upper_middle,-0.014260,0.084094,0.036703,0.035876,0.018704,0.053656,0.0,0.0,0.0,-0.014260,0.167598,0.038560,0.036412,0.019947,0.057251,0.0,0.0,0.0,-0.014260,0.202251,0.045662,0.043486,0.022612,0.070980,0.000000,0.000000,0.000000,-0.020666,1.252216,0.057616,0.042946,0.021713,0.106789,0.000000,0.000000,0.000000
578,UG,2016,580,2,-0.284113,30.363763,GPS,9999.0,1415.0,WGS84,12.753571,lower_middle,0.002042,0.404447,0.072845,0.036316,0.012776,0.168173,0.0,0.0,0.0,-0.003303,0.404447,0.059146,0.030262,0.011610,0.131903,0.0,0.0,0.0,-0.010216,0.404447,0.034194,0.026251,0.008897,0.067058,0.000000,0.000000,0.000000,-0.020321,1.935503,0.032801,0.019433,0.006110,0.068489,0.000000,0.000000,0.000000
215,UG,2016,216,2,1.278674,33.802150,GPS,9999.0,1061.0,WGS84,8.330000,poor,-0.049114,0.028838,0.003328,0.006539,-0.016914,0.020109,0.0,0.0,0.0,-0.049114,0.028838,0.003826,0.005558,-0.014727,0.020316,0.0,0.0,0.0,-0.066093,0.058433,0.001112,0.001693,-0.016922,0.018418,0.000000,0.000000,0.000000,-0.066093,0.152317,0.003029,0.002433,-0.016667,0.022442,0.000000,0.000000,0.000000
374,UG,2016,375,2,1.969558,33.072126,GPS,9999.0,1042.0,WGS84,9.859259,poor,-0.008384,0.079809,0.023059,0.023996,0.003695,0.042315,0.0,0.0,0.0,-0.010375,0.084071,0.021433,0.019772,0.002449,0.042133,0.0,0.0,0.0,-0.048790,0.118772,0.011131,0.009253,-0.006982,0.030520,0.000000,0.000000,0.000000,-0.048790,0.599718,0.009389,0.009697,-0.009458,0.029202,0.000000,0.000000,0.000000
88,UG,2016,89,1,0.000000,0.000000,MIS,9999.0,9999.0,WGS84,46.334483,rich,-0.028130,0.044645,0.012234,0.011225,-0.001151,0.025781,0.0,0.0,0.0,-0.028130,0.044645,0.010716,0.010429,-0.004052,0.025350,0.0,0.0,0.0,-0.035058,0.046851,0.010440,0.010444,-0.003764,0.024393,0.000000,0.000000,0.000000,-0.044665,0.262196,0.010108,0.010734,-0.005217,0.025383,0.000000,0.000000,0.000000
604,UG,2016,606,1,-0.056742,30.430154,GPS,9999.0,1366.0,WGS84,15.189655,upper_middle,-0.001145,0.055418,0.028572,0.029492,0.015446,0.041506,0.0,0.0,0.0,-0.006559,0.055418,0.026679,0.026312,0.012078,0.040741,0.0,0.0,0.0,-0.008361,0.156578,0.030480,0.029854,0.013078,0.048651,0.000000,0.000000,0.000000,-0.020653,1.850205,0.052143,0.027431,0.013458,0.110284,0.000000,0.000000,0.000000
563,UG,2016,565,2,-0.292785,30.618511,GPS,9999.0,1391.0,WGS84,12.734483,lower_middle,0.006233,0.058289,0.035643,0.033705,0.024472,0.047601,0.0,0.0,0.0,0.006233,0.070477,0.035538,0.033818,0.023733,0.047999,0.0,0.0,0.0,-0.002963,14.328595,0.147203,0.033887,0.033569,0.367477,0.002032,0.002017,0.197760,-0.004922,14.328595,0.070678,0.032330,0.031777,0.145655,0.000508,0.000507,0.102969
670,UG,2016,672,2,0.046006,33.007856,GPS,9999.0,1142.0,WGS84,9.773077,poor,-0.019134,0.048886,0.013476,0.013919,-0.001503,0.027471,0.0,0.0,0.0,-0.019134,0.048886,0.011362,0.012839,-0.004155,0.025846,0.0,0.0,0.0,-0.040141,0.071892,0.006901,0.006159,-0.008686,0.022913,0.000000,0.000000,0.000000,-0.066123,0.080743,0.011297,0.008331,-0.009215,0.033984,0.000000,0.000000,0.000000
746,UG,2018,51,1,0.604643,31.393991,GPS,1277.0,1282.0,WGS84,20.129630,rich,0.223333,0.393